In [ ]:
# # ============================================
# # 0) IMPORTS
# # ============================================
# import os
# import glob
# import pandas as pd
# import numpy as np
# import shapely.wkt
# import shapely.geometry
# import geopandas as gpd
# from affine import Affine
# import rasterio.features
# from tqdm import tqdm  # Progress bar
# import gc  # Garbage Collector for memory management

# import ee

# # ============================================
# # 1) CONFIGURATION
# # ============================================

# # --- INPUT: The consolidated PKL from step 02a ---
# INPUT_PKL = r"D:\Development\RESEARCH\urban_flood_database\chronicle\hydromerit_pluvial_outputs\chronicle_df_with_pfdi_FULL.pkl"

# # --- OUTPUT: Where rain data will be saved ---
# OUT_DIR = r"D:\Development\RESEARCH\urban_flood_database\chronicle\imerg_rain_outputs"
# OUT_FINAL_PKL = os.path.join(OUT_DIR, "chronicle_urban_df_with_IMERG_FULL.pkl")

# # IMERG Constants
# # Data available from June 2000
# IMERG_START_DATE = pd.Timestamp("2000-06-01") 
# SCALE = 0.1  # 0.1 Degree resolution
# CRS = 'EPSG:4326'

# # Batch Settings
# # Keeping it safe at 100 to avoid memory overflow with 3D arrays
# BATCH_SIZE = 1000 
# N_BATCHES_TO_RUN = 350  # Limit execution to 30 batches (3000 events)

# # ============================================
# # 2) HELPERS
# # ============================================

# def ensure_out_dir(path):
#     """Create output directory if it doesn't exist."""
#     if not os.path.exists(path):
#         os.makedirs(path, exist_ok=True)

# def initialize_ee():
#     """Initialize Earth Engine."""
#     try:
#         ee.Initialize()
#         print("Earth Engine initialized.")
#     except Exception:
#         print("Authenticating Earth Engine...")
#         ee.Authenticate()
#         ee.Initialize()
#         print("Earth Engine initialized after auth.")

# def get_next_batch_index(out_dir):
#     """
#     Scans output directory for 'imerg_batch_XXXX.pkl' to determine 
#     the next batch number for file naming.
#     """
#     if not os.path.exists(out_dir):
#         return 0
    
#     pattern = os.path.join(out_dir, "imerg_batch_*.pkl")
#     files = glob.glob(pattern)
    
#     if not files:
#         return 0
    
#     max_batch = -1
#     for f in files:
#         try:
#             filename = os.path.basename(f)
#             # filename format: imerg_batch_0001.pkl
#             num_part = filename.split('_')[-1].split('.')[0]
#             batch_num = int(num_part)
#             if batch_num > max_batch:
#                 max_batch = batch_num
#         except ValueError:
#             continue
            
#     return max_batch + 1

# def extract_rain_data(event_row):
#     """
#     Extracts IMERG rain matrix, metadata, and mask for a single event.
    
#     Time Window Logic:
#     - Start: 72 hours BEFORE event start.
#     - End: 24 hours AFTER event end.
    
#     Returns: (rain_matrix, rain_mask, rain_meta)
#     """
#     # 1. Geometry Setup
#     try:
#         # Load geometry from WKT
#         if isinstance(event_row['geometry_wkt'], str):
#             poly_geom = shapely.wkt.loads(event_row['geometry_wkt'])
#         else:
#             # Fallback if it's already a geometry object
#             poly_geom = event_row['geometry_wkt']
            
#         bounds = poly_geom.bounds # (minx, miny, maxx, maxy)
#         roi = ee.Geometry.BBox(*bounds)
        
#         # 2. Time Window Calculation
#         # UPDATED: Taking 72 hours prior to the start time
#         start_t = event_row['start_time'] - pd.Timedelta(hours=72)
#         # We keep a buffer after the end time as well (e.g., 24 hours)
#         end_t = event_row['end_time'] + pd.Timedelta(hours=24) 
        
#         # 3. GEE Request
#         imerg_coll = ee.ImageCollection("NASA/GPM_L3/IMERG_V06") \
#             .select('precipitationCal') \
#             .filterBounds(roi) \
#             .filterDate(start_t, end_t)
        
#         # Check if collection is empty
#         if imerg_coll.size().getInfo() == 0:
#             return None, None, None

#         # 4. Download Data (sampleRectangle)
#         # Convert collection to a multi-band image (each band is a time step)
#         stack = imerg_coll.toBands()
        
#         # sampleRectangle downloads the raw pixels within the BBox
#         pixel_dict = stack.sampleRectangle(region=roi).getInfo()
#         properties = pixel_dict['properties']
        
#         # 5. Parse & Stack Arrays
#         # Keys are like '20000604120000_precipitationCal'
#         band_keys = sorted(list(properties.keys()))
        
#         arrays_list = []
#         for b in band_keys:
#             # Convert list to numpy array (float32 to save memory)
#             arr = np.array(properties[b], dtype=np.float32)
#             arrays_list.append(arr)
            
#         # Stack into 3D Array: (Time, Height, Width)
#         rain_matrix = np.stack(arrays_list)
        
#         # 6. Create Metadata (Anchor)
#         height, width = rain_matrix.shape[1], rain_matrix.shape[2]
#         min_lon, min_lat, max_lon, max_lat = bounds
        
#         # Transform for Rasterio (Lat/Lon)
#         # Note: We align to the BBox top-left
#         transform = Affine(SCALE, 0, min_lon, 0, -SCALE, max_lat)
        
#         meta = {
#             'origin_top_left': (max_lat, min_lon), # (Lat, Lon)
#             'scale': SCALE,
#             'shape': (height, width),
#             'timestamps': band_keys # Store timestamp keys to map matrix layers to time
#         }

#         # 7. Create Binary Mask (Polygon shape on grid)
#         # 1 = Inside Polygon, 0 = Outside
#         mask = rasterio.features.rasterize(
#             [(poly_geom, 1)],
#             out_shape=(height, width),
#             transform=transform,
#             fill=0,
#             all_touched=True,
#             dtype=np.uint8
#         )
        
#         return rain_matrix, mask, meta

#     except Exception as e:
#         # If extraction fails (e.g., GEE timeout), return None
#         return None, None, None

In [ ]:
# # ============================================
# # 3) INITIALIZATION
# # ============================================
# initialize_ee()
# ensure_out_dir(OUT_DIR)

# # Load Input Data (The consolidated file from 02a)
# print(f"Loading full dataset: {INPUT_PKL}")
# if not os.path.exists(INPUT_PKL):
#     raise FileNotFoundError(f"Input file not found: {INPUT_PKL}. Please run 02a first.")

# df = pd.read_pickle(INPUT_PKL)

# # Basic cleaning
# df = df.replace([np.inf, -np.inf], np.nan)
# df['start_time'] = pd.to_datetime(df['start_time'], unit='s')
# df['end_time'] = pd.to_datetime(df['end_time'], unit='s')

# # Filter for IMERG Era (Post June 2000)
# # Events before this date will not have IMERG data
# df_valid = df[df['start_time'] >= IMERG_START_DATE].copy()

# print(f"Total events in input: {len(df)}")
# print(f"Events valid for IMERG (post-2000): {len(df_valid)}")

# # ============================================
# # 4) SMART BATCH PROCESSING (ID-BASED)
# # ============================================

# print(f"--- PREPARING WORK PLAN ---")

# # 1. Identify what is already done
# # We scan the output directory for existing RAIN batches to avoid re-processing
# processed_ids = set()
# pkl_pattern = os.path.join(OUT_DIR, "imerg_batch_*.pkl")
# existing_files = glob.glob(pkl_pattern)

# if existing_files:
#     print(f"Found {len(existing_files)} existing batch files. Scanning for processed IDs...")
#     for f in tqdm(existing_files, desc="Indexing existing data"):
#         try:
#             # Only read columns needed for ID check to save memory
#             df_temp = pd.read_pickle(f)
#             if 'event_id' in df_temp.columns:
#                 processed_ids.update(df_temp['event_id'].tolist())
#             del df_temp
#         except Exception as e:
#             print(f"Skipping corrupted file {f}: {e}")

# print(f"Total events already processed: {len(processed_ids)}")

# # 2. Filter the Main DataFrame
# # We keep only rows whose ID is NOT in the processed set
# df_todo = df_valid[~df_valid['event_id'].isin(processed_ids)].copy()

# print(f"Events remaining to process: {len(df_todo)}")

# if len(df_todo) == 0:
#     print("All events are already processed! Nothing to do.")
# else:
#     # 3. Process the remaining rows in new batches
#     start_batch_num = get_next_batch_index(OUT_DIR)
#     n_remaining = len(df_todo)
    
#     # Calculate stop limit based on N_BATCHES_TO_RUN
#     max_rows_limit = N_BATCHES_TO_RUN * BATCH_SIZE
#     stop_at_row = min(n_remaining, max_rows_limit)

#     print(f"Plan: Processing {min(N_BATCHES_TO_RUN, n_remaining // BATCH_SIZE + 1)} batches.")
    
#     # Iterate in chunks up to stop_at_row
#     for batch_i in range(0, stop_at_row, BATCH_SIZE):
        
#         # Determine actual batch number for filename
#         current_file_num = start_batch_num + (batch_i // BATCH_SIZE)
        
#         # Slice the TODO dataframe
#         batch_df = df_todo.iloc[batch_i : batch_i + BATCH_SIZE].copy()
        
#         print(f"\nProcessing Batch {current_file_num} ({len(batch_df)} events)...")
        
#         matrices = []
#         masks = []
#         metas = []
        
#         # Inner loop: iterate rows in current batch
#         for idx, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc=f"Batch {current_file_num}"):
#             mat, msk, mt = extract_rain_data(row)
#             matrices.append(mat)
#             masks.append(msk)
#             metas.append(mt)
        
#         # Assign results to columns
#         batch_df['imerg_matrix'] = matrices
#         batch_df['imerg_mask'] = masks
#         batch_df['imerg_meta'] = metas
        
#         # Save batch to disk (Pickle)
#         out_path = os.path.join(OUT_DIR, f"imerg_batch_{current_file_num:04d}.pkl")
#         batch_df.to_pickle(out_path)
#         print(f"Saved: {out_path}")
        
#         # === MEMORY CLEANUP ===
#         # Explicitly delete large objects to free RAM for next iteration
#         del batch_df
#         del matrices
#         del masks
#         del metas
#         gc.collect() # Force garbage collection
#         # ======================

#     print("\n--- BATCH LIMIT REACHED ---")
#     print(f"Stopped execution after {N_BATCHES_TO_RUN} batches as requested.")

In [ ]:
# # ============================================
# # 5) FINAL MERGE & CLEANUP
# # ============================================
# print("\n--- FINALIZING ---")
# print("Merging all batch files...")

# pkl_pattern = os.path.join(OUT_DIR, "imerg_batch_*.pkl")
# all_pkl_files = glob.glob(pkl_pattern)

# if not all_pkl_files:
#     print("No output files found.")
# else:
#     # Concatenate all batches
#     df_list = []
#     for f in tqdm(all_pkl_files, desc="Loading Batches"):
#         try:
#             df_list.append(pd.read_pickle(f))
#         except Exception as e:
#             print(f"Error loading {f}: {e}")
            
#     if df_list:
#         # This DataFrame contains ONLY the processed events with rain data
#         df_results = pd.concat(df_list, ignore_index=True)
        
#         print(f"Loaded {len(df_results)} processed rain events (raw count).")
#         print("Merging results back to main dataset structure...")
        
#         # Reload the original input again to ensure we have the clean base columns
#         df_base = pd.read_pickle(INPUT_PKL)
        
#         # Merge: 'inner' keeps only keys that appear in BOTH DataFrames
#         df_final = df_base.merge(
#             df_results[['event_id', 'imerg_matrix', 'imerg_mask', 'imerg_meta']], 
#             on='event_id', 
#             how='inner' 
#         )
        
#         # --- NEW: FILTER OUT NULL RAIN DATA ---
        
#         # 1. Ensure start_time is datetime format for readable printing
#         if 'start_time' in df_final.columns:
#             df_final['start_time'] = pd.to_datetime(df_final['start_time'], unit='s')

#         # 2. Identify rows where 'imerg_matrix' is Null/None (Values of 0 are NOT null, so they stay)
#         missing_rain_mask = df_final['imerg_matrix'].isnull()
#         missing_events = df_final[missing_rain_mask]
        
#         # 3. Print dates of dropped events
#         if not missing_events.empty:
#             print(f"\n[WARNING] Found {len(missing_events)} events with NULL rain data. Removing them...")
#             print("--- Dropped Events Log ---")
#             for idx, row in missing_events.iterrows():
#                 try:
#                     d_str = row['start_time'].strftime('%d-%m-%Y %H:%M')
#                 except:
#                     d_str = str(row['start_time'])
#                 print(f"Removing ID: {row['event_id']} | Date: {d_str}")
            
#             # 4. Perform the drop
#             df_final = df_final[~missing_rain_mask].copy()
#             print(f"--- Cleaned. Remaining events: {len(df_final)} ---")
#         else:
#             print("No NULL rain events found. All processed events are valid.")

#         # --------------------------------------

#         # Save Final PKL (Cleaned subset)
#         df_final.to_pickle(OUT_FINAL_PKL)
        
#         print(f"SUCCESS! Final dataset saved to: {OUT_FINAL_PKL}")
        
#         # Final Verification
#         if 'imerg_matrix' in df_final.columns:
#             count = df_final['imerg_matrix'].notnull().sum()
#             print(f"Verified valid Rain Data events: {count}")
#     else:
#         print("Failed to load any batch files.")

In [4]:
# ============================================
# 0) IMPORTS
# ============================================
import os
import glob
import pandas as pd
import numpy as np
import shapely.wkt
import shapely.geometry
import geopandas as gpd
from affine import Affine
import rasterio.features
from tqdm import tqdm  # Progress bar
import gc  # Garbage Collector for memory management

import ee

# ============================================
# 1) CONFIGURATION
# ============================================

# --- INPUT: The consolidated PKL from step 02a ---
INPUT_PKL = r"D:\Development\RESEARCH\urban_flood_database\chronicle\hydromerit_pluvial_outputs\chronicle_df_with_pfdi_FULL.pkl"

# --- OUTPUT: Where rain data will be saved ---
OUT_DIR = r"D:\Development\RESEARCH\urban_flood_database\chronicle\imerg_rain_outputs"
OUT_FINAL_PKL = os.path.join(OUT_DIR, "chronicle_urban_df_with_IMERG_FULL.pkl")

# IMERG Constants
# Data available from June 2000
IMERG_START_DATE = pd.Timestamp("2000-06-01") 
SCALE = 0.1  # 0.1 Degree resolution
CRS = 'EPSG:4326'

# Batch Settings
BATCH_SIZE = 1000 
N_BATCHES_TO_RUN = 350  

# ============================================
# 2) HELPERS
# ============================================

def ensure_out_dir(path):
    """Create output directory if it doesn't exist."""
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def initialize_ee():
    """Initialize Earth Engine."""
    try:
        ee.Initialize()
        print("Earth Engine initialized.")
    except Exception:
        print("Authenticating Earth Engine...")
        ee.Authenticate()
        ee.Initialize()
        print("Earth Engine initialized after auth.")

def get_next_batch_index(out_dir):
    """
    Scans output directory for 'imerg_batch_XXXX.pkl' to determine 
    the next batch number for file naming.
    """
    if not os.path.exists(out_dir):
        return 0
    
    pattern = os.path.join(out_dir, "imerg_batch_*.pkl")
    files = glob.glob(pattern)
    
    if not files:
        return 0
    
    max_batch = -1
    for f in files:
        try:
            filename = os.path.basename(f)
            num_part = filename.split('_')[-1].split('.')[0]
            batch_num = int(num_part)
            if batch_num > max_batch:
                max_batch = batch_num
        except ValueError:
            continue
            
    return max_batch + 1

def extract_rain_data(event_row):
    """
    Extracts IMERG rain matrix, metadata, and mask for a single event.
    Attempts to use FINAL (V06) collection first.
    Falls back to LATE collection if FINAL is unavailable.
    
    Returns: (rain_matrix, rain_mask, rain_meta, imerg_type)
    """
    try:
        # 1. Geometry Setup
        if isinstance(event_row['geometry_wkt'], str):
            poly_geom = shapely.wkt.loads(event_row['geometry_wkt'])
        else:
            poly_geom = event_row['geometry_wkt']
            
        bounds = poly_geom.bounds # (minx, miny, maxx, maxy)
        roi = ee.Geometry.BBox(*bounds)
        
        # 2. Time Window Calculation
        start_t = event_row['start_time'] - pd.Timedelta(hours=72)
        end_t = event_row['end_time'] + pd.Timedelta(hours=24) 
        
        # 3. GEE Request with Fallback Logic
        imerg_type = 'final'
        imerg_coll = ee.ImageCollection("NASA/GPM_L3/IMERG_V06") \
            .select('precipitationCal') \
            .filterBounds(roi) \
            .filterDate(start_t, end_t)
        
        # Fallback to LATE collection if FINAL run is empty (common for recent dates)
        if imerg_coll.size().getInfo() == 0:
            imerg_type = 'late'
            imerg_coll = ee.ImageCollection("NASA/GPM_L3/IMERG_V06_LATE") \
                .select('precipitationCal') \
                .filterBounds(roi) \
                .filterDate(start_t, end_t)

        # Check if data exists in either collection
        if imerg_coll.size().getInfo() == 0:
            return None, None, None, 'missing'

        # 4. Download Data
        stack = imerg_coll.toBands()
        pixel_dict = stack.sampleRectangle(region=roi).getInfo()
        properties = pixel_dict['properties']
        
        # 5. Parse & Stack Arrays
        band_keys = sorted(list(properties.keys()))
        arrays_list = [np.array(properties[b], dtype=np.float32) for b in band_keys]
        rain_matrix = np.stack(arrays_list)
        
        # 6. Create Metadata
        height, width = rain_matrix.shape[1], rain_matrix.shape[2]
        min_lon, min_lat, max_lon, max_lat = bounds
        transform = Affine(SCALE, 0, min_lon, 0, -SCALE, max_lat)
        
        meta = {
            'origin_top_left': (max_lat, min_lon),
            'scale': SCALE,
            'shape': (height, width),
            'timestamps': band_keys
        }

        # 7. Create Binary Mask
        mask = rasterio.features.rasterize(
            [(poly_geom, 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            all_touched=True,
            dtype=np.uint8
        )
        
        return rain_matrix, mask, meta, imerg_type

    except Exception as e:
        # Return error status to avoid breaking the batch
        print(f"Error processing ID {event_row.get('event_id')}: {e}")
        return None, None, None, 'error'

In [ ]:
# ============================================
# 3) INITIALIZATION
# ============================================
initialize_ee()
ensure_out_dir(OUT_DIR)

print(f"Loading full dataset: {INPUT_PKL}")
if not os.path.exists(INPUT_PKL):
    raise FileNotFoundError(f"Input file not found: {INPUT_PKL}.")

df = pd.read_pickle(INPUT_PKL)
df = df.replace([np.inf, -np.inf], np.nan)
df['start_time'] = pd.to_datetime(df['start_time'], unit='s')
df['end_time'] = pd.to_datetime(df['end_time'], unit='s')

# Filter for IMERG Era
df_valid = df[df['start_time'] >= IMERG_START_DATE].copy()

print(f"Total events in input: {len(df)}")
print(f"Events valid for IMERG (post-2000): {len(df_valid)}")

# ============================================
# 4) SMART BATCH PROCESSING (ID-BASED)
# ============================================

print(f"--- PREPARING WORK PLAN ---")

processed_ids = set()
pkl_pattern = os.path.join(OUT_DIR, "imerg_batch_*.pkl")
existing_files = glob.glob(pkl_pattern)

if existing_files:
    print(f"Found {len(existing_files)} existing batch files. Scanning for processed IDs...")
    for f in tqdm(existing_files, desc="Indexing existing data"):
        try:
            df_temp = pd.read_pickle(f)
            if 'event_id' in df_temp.columns:
                processed_ids.update(df_temp['event_id'].tolist())
            del df_temp
        except Exception as e:
            print(f"Skipping corrupted file {f}: {e}")

print(f"Total events already processed: {len(processed_ids)}")

df_todo = df_valid[~df_valid['event_id'].isin(processed_ids)].copy()
print(f"Events remaining to process: {len(df_todo)}")

if len(df_todo) == 0:
    print("All events are already processed! Nothing to do.")
else:
    start_batch_num = get_next_batch_index(OUT_DIR)
    n_remaining = len(df_todo)
    max_rows_limit = N_BATCHES_TO_RUN * BATCH_SIZE
    stop_at_row = min(n_remaining, max_rows_limit)

    print(f"Plan: Processing {min(N_BATCHES_TO_RUN, n_remaining // BATCH_SIZE + 1)} batches.")
    
    for batch_i in range(0, stop_at_row, BATCH_SIZE):
        current_file_num = start_batch_num + (batch_i // BATCH_SIZE)
        batch_df = df_todo.iloc[batch_i : batch_i + BATCH_SIZE].copy()
        
        print(f"\nProcessing Batch {current_file_num} ({len(batch_df)} events)...")
        
        matrices, masks, metas, types = [], [], [], []
        
        for idx, row in tqdm(batch_df.iterrows(), total=len(batch_df), desc=f"Batch {current_file_num}"):
            mat, msk, mt, itype = extract_rain_data(row)
            matrices.append(mat)
            masks.append(msk)
            metas.append(mt)
            types.append(itype)
        
        # Assign results including the new source type column
        batch_df['imerg_matrix'] = matrices
        batch_df['imerg_mask'] = masks
        batch_df['imerg_meta'] = metas
        batch_df['imerg_type'] = types
        
        out_path = os.path.join(OUT_DIR, f"imerg_batch_{current_file_num:04d}.pkl")
        batch_df.to_pickle(out_path)
        print(f"Saved: {out_path}")
        
        del batch_df, matrices, masks, metas, types
        gc.collect()

    print("\n--- BATCH LIMIT REACHED ---")

In [6]:
# ============================================
# 5) FINAL MERGE & CLEANUP
# ============================================
print("\n--- FINALIZING ---")
print("Merging all batch files...")

pkl_pattern = os.path.join(OUT_DIR, "imerg_batch_*.pkl")
all_pkl_files = glob.glob(pkl_pattern)

if not all_pkl_files:
    print("No output files found.")
else:
    # Concatenate all batches
    df_list = []
    for f in tqdm(all_pkl_files, desc="Loading Batches"):
        try:
            df_list.append(pd.read_pickle(f))
        except Exception as e:
            print(f"Error loading {f}: {e}")
            
    if df_list:
        # Create results dataframe from all batches
        df_results = pd.concat(df_list, ignore_index=True)
        
        # Determine columns to merge
        merge_cols = ['event_id', 'imerg_matrix', 'imerg_mask', 'imerg_meta']
        if 'imerg_type' in df_results.columns:
            merge_cols.append('imerg_type')
        
        print(f"Loaded {len(df_results)} processed rain events.")
        print("Merging results back to main dataset structure...")
        
        # Reload the original input to ensure clean base columns
        df_base = pd.read_pickle(INPUT_PKL)
        
        # Merge: 'inner' keeps only keys that appear in BOTH DataFrames
        df_final = df_base.merge(
            df_results[merge_cols], 
            on='event_id', 
            how='inner' 
        )
        
        # --- HANDLING OLD BATCHES (Backfilling) ---
        if 'imerg_type' in df_final.columns:
            df_final['imerg_type'] = df_final['imerg_type'].fillna('final')
        else:
            df_final['imerg_type'] = 'final'

        # Ensure start_time is datetime for readable printing
        if 'start_time' in df_final.columns:
            if not pd.api.types.is_datetime64_any_dtype(df_final['start_time']):
                df_final['start_time'] = pd.to_datetime(df_final['start_time'], unit='s')

        # --- FILTER OUT NULL RAIN DATA & LOG DROPPED EVENTS ---
        
        # Identify rows where 'imerg_matrix' is Null (due to GEE masked pixels or errors)
        missing_rain_mask = df_final['imerg_matrix'].isnull()
        missing_events = df_final[missing_rain_mask].copy()
        
        if not missing_events.empty:
            print(f"\n[!] Found {len(missing_events)} events with NULL rain data. These will be removed.")
            print("--- List of Dropped Events ---")
            
            # Print each dropped event with its ID, date and type (if available)
            for idx, row in missing_events.iterrows():
                try:
                    date_str = row['start_time'].strftime('%Y-%m-%d %H:%M')
                except:
                    date_str = str(row['start_time'])
                
                # Check for error type if it was logged during extraction
                e_type = row.get('imerg_type', 'unknown')
                print(f"ID: {row['event_id']} | Date: {date_str} | Status: {e_type}")
            
            print("------------------------------")
            
            # Perform the drop
            df_final = df_final[~missing_rain_mask].copy()
            print(f"--- Cleaned. Remaining events: {len(df_final)} ---")
        else:
            print("No NULL rain events found. All processed events are valid.")

        # Save Final PKL
        df_final.to_pickle(OUT_FINAL_PKL)
        print(f"\nSUCCESS! Final dataset saved to: {OUT_FINAL_PKL}")
        
        # --- FINAL SUMMARY STATISTICS ---
        print("\n--- IMERG Data Quality Summary ---")
        if 'imerg_type' in df_final.columns:
            stats = df_final['imerg_type'].value_counts()
            for label, count in stats.items():
                print(f" - {label.upper()}: {count} events")
        
        # Final Verification
        count = df_final['imerg_matrix'].notnull().sum()
        print(f"Total verified valid Rain Data events: {count}")
    else:
        print("Failed to load any batch files.")


--- FINALIZING ---
Merging all batch files...


Loading Batches: 100%|███████████████████████████████████████████████████████████████| 793/793 [00:37<00:00, 21.15it/s]


Loaded 752500 processed rain events.
Merging results back to main dataset structure...

[!] Found 51588 events with NULL rain data. These will be removed.
--- List of Dropped Events ---
ID: 1134 | Date: 2000-11-01 00:00 | Status: final
ID: 1546 | Date: 2000-12-06 00:00 | Status: final
ID: 1639 | Date: 2000-12-13 00:00 | Status: final
ID: 1668 | Date: 2001-01-01 00:00 | Status: final
ID: 1724 | Date: 2001-01-01 00:00 | Status: final
ID: 1788 | Date: 2001-01-01 00:00 | Status: final
ID: 1984 | Date: 2001-05-01 00:00 | Status: final
ID: 2697 | Date: 2002-01-01 00:00 | Status: final
ID: 4777 | Date: 2003-09-02 00:00 | Status: final
ID: 4778 | Date: 2003-09-02 00:00 | Status: final
ID: 4779 | Date: 2003-09-02 00:00 | Status: final
ID: 4780 | Date: 2003-09-02 00:00 | Status: final
ID: 4783 | Date: 2003-09-02 00:00 | Status: final
ID: 4784 | Date: 2003-09-02 00:00 | Status: final
ID: 4785 | Date: 2003-09-02 00:00 | Status: final
ID: 4786 | Date: 2003-09-02 00:00 | Status: final
ID: 4788 | Dat

ID: 709100 | Date: 2024-06-24 00:00 | Status: error
ID: 709101 | Date: 2024-06-24 00:00 | Status: error
ID: 709102 | Date: 2024-06-24 00:00 | Status: error
ID: 709103 | Date: 2024-06-24 00:00 | Status: error
ID: 709104 | Date: 2024-06-24 00:00 | Status: error
ID: 709105 | Date: 2024-06-24 00:00 | Status: error
ID: 709106 | Date: 2024-06-25 00:00 | Status: error
ID: 709107 | Date: 2024-06-25 00:00 | Status: error
ID: 709108 | Date: 2024-06-25 00:00 | Status: error
ID: 709109 | Date: 2024-06-25 00:00 | Status: error
ID: 709110 | Date: 2024-06-25 00:00 | Status: error
ID: 709111 | Date: 2024-06-25 00:00 | Status: error
ID: 709112 | Date: 2024-06-25 00:00 | Status: error
ID: 709113 | Date: 2024-06-25 00:00 | Status: error
ID: 709114 | Date: 2024-06-25 00:00 | Status: error
ID: 709115 | Date: 2024-06-25 00:00 | Status: error
ID: 709116 | Date: 2024-06-25 00:00 | Status: error
ID: 709117 | Date: 2024-06-25 00:00 | Status: error
ID: 709118 | Date: 2024-06-25 00:00 | Status: error
ID: 709119 |

ID: 718779 | Date: 2024-07-12 00:00 | Status: error
ID: 718780 | Date: 2024-07-12 00:00 | Status: error
ID: 718781 | Date: 2024-07-12 00:00 | Status: error
ID: 718782 | Date: 2024-07-12 00:00 | Status: error
ID: 718783 | Date: 2024-07-12 00:00 | Status: error
ID: 718784 | Date: 2024-07-12 00:00 | Status: error
ID: 718785 | Date: 2024-07-12 00:00 | Status: error
ID: 718786 | Date: 2024-07-12 00:00 | Status: error
ID: 718787 | Date: 2024-07-12 00:00 | Status: error
ID: 718788 | Date: 2024-07-12 00:00 | Status: error
ID: 718789 | Date: 2024-07-12 00:00 | Status: error
ID: 718790 | Date: 2024-07-12 00:00 | Status: error
ID: 718791 | Date: 2024-07-12 00:00 | Status: error
ID: 718792 | Date: 2024-07-12 00:00 | Status: error
ID: 718793 | Date: 2024-07-12 00:00 | Status: error
ID: 718794 | Date: 2024-07-12 00:00 | Status: error
ID: 718795 | Date: 2024-07-12 00:00 | Status: error
ID: 718796 | Date: 2024-07-12 00:00 | Status: error
ID: 718797 | Date: 2024-07-12 00:00 | Status: error
ID: 718798 |

ID: 728446 | Date: 2024-08-01 00:00 | Status: error
ID: 728447 | Date: 2024-08-01 00:00 | Status: error
ID: 728448 | Date: 2024-08-01 00:00 | Status: error
ID: 728449 | Date: 2024-08-01 00:00 | Status: error
ID: 728450 | Date: 2024-08-01 00:00 | Status: error
ID: 728451 | Date: 2024-08-01 00:00 | Status: error
ID: 728452 | Date: 2024-08-01 00:00 | Status: error
ID: 728453 | Date: 2024-08-01 00:00 | Status: error
ID: 728454 | Date: 2024-08-01 00:00 | Status: error
ID: 728455 | Date: 2024-08-01 00:00 | Status: error
ID: 728456 | Date: 2024-08-01 00:00 | Status: error
ID: 728457 | Date: 2024-08-01 00:00 | Status: error
ID: 728458 | Date: 2024-08-01 00:00 | Status: error
ID: 728459 | Date: 2024-08-01 00:00 | Status: error
ID: 728460 | Date: 2024-08-01 00:00 | Status: error
ID: 728461 | Date: 2024-08-01 00:00 | Status: error
ID: 728462 | Date: 2024-08-01 00:00 | Status: error
ID: 728463 | Date: 2024-08-01 00:00 | Status: error
ID: 728464 | Date: 2024-08-01 00:00 | Status: error
ID: 728465 |

ID: 738270 | Date: 2024-08-27 00:00 | Status: error
ID: 738271 | Date: 2024-08-27 00:00 | Status: error
ID: 738272 | Date: 2024-08-27 00:00 | Status: error
ID: 738273 | Date: 2024-08-27 00:00 | Status: error
ID: 738274 | Date: 2024-08-27 00:00 | Status: error
ID: 738275 | Date: 2024-08-27 00:00 | Status: error
ID: 738276 | Date: 2024-08-27 00:00 | Status: error
ID: 738277 | Date: 2024-08-27 00:00 | Status: error
ID: 738278 | Date: 2024-08-27 00:00 | Status: error
ID: 738279 | Date: 2024-08-27 00:00 | Status: error
ID: 738280 | Date: 2024-08-27 00:00 | Status: error
ID: 738281 | Date: 2024-08-27 00:00 | Status: error
ID: 738282 | Date: 2024-08-27 00:00 | Status: error
ID: 738283 | Date: 2024-08-27 00:00 | Status: error
ID: 738284 | Date: 2024-08-27 00:00 | Status: error
ID: 738285 | Date: 2024-08-27 00:00 | Status: error
ID: 738286 | Date: 2024-08-27 00:00 | Status: error
ID: 738287 | Date: 2024-08-27 00:00 | Status: error
ID: 738288 | Date: 2024-08-27 00:00 | Status: error
ID: 738289 |

ID: 748006 | Date: 2024-09-14 00:00 | Status: error
ID: 748007 | Date: 2024-09-14 00:00 | Status: error
ID: 748008 | Date: 2024-09-14 00:00 | Status: error
ID: 748009 | Date: 2024-09-14 00:00 | Status: error
ID: 748010 | Date: 2024-09-14 00:00 | Status: error
ID: 748011 | Date: 2024-09-14 00:00 | Status: error
ID: 748012 | Date: 2024-09-14 00:00 | Status: error
ID: 748013 | Date: 2024-09-14 00:00 | Status: error
ID: 748014 | Date: 2024-09-14 00:00 | Status: error
ID: 748015 | Date: 2024-09-14 00:00 | Status: error
ID: 748016 | Date: 2024-09-14 00:00 | Status: error
ID: 748017 | Date: 2024-09-14 00:00 | Status: error
ID: 748018 | Date: 2024-09-14 00:00 | Status: error
ID: 748019 | Date: 2024-09-14 00:00 | Status: error
ID: 748020 | Date: 2024-09-14 00:00 | Status: error
ID: 748021 | Date: 2024-09-14 00:00 | Status: error
ID: 748022 | Date: 2024-09-14 00:00 | Status: error
ID: 748023 | Date: 2024-09-14 00:00 | Status: error
ID: 748024 | Date: 2024-09-14 00:00 | Status: error
ID: 748025 |

--- Cleaned. Remaining events: 700912 ---

SUCCESS! Final dataset saved to: D:\Development\RESEARCH\urban_flood_database\chronicle\imerg_rain_outputs\chronicle_urban_df_with_IMERG_FULL.pkl

--- IMERG Data Quality Summary ---
 - FINAL: 700912 events
Total verified valid Rain Data events: 700912


In [10]:
# Check the distribution of IMERG data quality
print(df_final['imerg_type'].value_counts())

final    700912
Name: imerg_type, dtype: int64


# HERE SHOULD BE COMPLEMENTERY CODE:
הקוד צריך לעבור שוב על כל האירועים שהוסרו מהבאצים כי לא היה בהם דאטא.
הוא יבחן שוב אם יש את הדאטא שלהם בגרסה הטובה ביותר ואם לא אז בגרסה השנייה הפחות טובה.
הרעיון - לודא שלא נפלו האירועים בגלל מגבלות גישה ועומס על הענן של גוגל בשלב ראשון, ואם לא זה אז האם יש בעיה בזמינות הנתונים עצמם בגרסה המירבית ואז זה יגש לגרסה הפחות טובה.


המלצת הגימיני:

# --- COMPLEMENTARY CLEANUP STEP ---
print("\n--- RUNNING COMPLEMENTARY RECOVERY ---")

# Load the reference (all events) and the current results
df_base = pd.read_pickle(INPUT_PKL)
df_final = pd.read_pickle(OUT_FINAL_PKL)

# Find IDs that are in base but NOT in final
missing_mask = ~df_base['event_id'].isin(df_final['event_id'])
df_missing = df_base[missing_mask].copy()

print(f"Status check: {len(df_missing)} events are currently missing from the final PKL.")

if len(df_missing) > 0:
    print(f"Attempting to recover {len(df_missing)} events...")
    recovered_list = []
    
    # Try one last time for each missing event
    for idx, row in tqdm(df_missing.iterrows(), total=len(df_missing), desc="Recovery"):
        # This will use the Fallback logic (Final -> Late)
        mat, msk, mt, itype = extract_rain_data(row)
        
        # Only keep if we actually found data this time
        if mat is not None:
            recovered_row = row.to_dict()
            recovered_row.update({
                'imerg_matrix': mat,
                'imerg_mask': msk,
                'imerg_meta': mt,
                'imerg_type': itype
            })
            recovered_list.append(recovered_row)
    
    if recovered_list:
        df_recovered = pd.DataFrame(recovered_list)
        # Combine the old final results with the newly recovered ones
        df_updated = pd.concat([df_final, df_recovered], ignore_index=True)
        
        # Final save
        df_updated.to_pickle(OUT_FINAL_PKL)
        print(f"Recovery successful! Added {len(recovered_list)} events.")
        print(f"Final count: {len(df_updated)} events.")
    else:
        print("No additional data could be recovered (all remaining events likely out of range or truly empty).")

In [ ]:
df_final

# לעדכן לגרסה V07 של imerg

In [ ]:
700912

In [8]:
print(ee.ImageCollection("NASA/GPM_L3/IMERG_V06").limit(1, "system:time_start", False).first().date().format().getInfo())

2024-06-02T18:30:00


In [13]:
print(ee.ImageCollection("NASA/GPM_L3/IMERG_V07").limit(1, "system:time_start", False).first().date().format().getInfo())

2026-02-14T03:30:00
